In [ ]:
from fastapi import FastAPI
from fastapi.responses import JSONResponse
import nest_asyncio
import uvicorn
import threading
import mlflow
from openai import OpenAI
from loguru import logger

# Apply nest_asyncio to allow nested event loops in Jupyter
nest_asyncio.apply()

In [ ]:
PORT = "5001"
EXPERIMENT_NAME = "FastAPI-Ollama"
OLLAMA_API_URL = "http://localhost:11434/v1"

LLM_MODEL = "gpt-oss:20b"
TEMPERATURE = 0.1
MAX_TOKENS = 100

In [ ]:
mlflow.openai.autolog()
mlflow.set_tracking_uri(f"http://localhost:{PORT}")
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
def set_llm(url):
    try:
        client = OpenAI(
            base_url=url,
            api_key="dummy",
        )
        return client
    except Exception as e:
        print(f"Error initializing OpenAI client: {e}")
        return None

client = set_llm(OLLAMA_API_URL)

In [ ]:
def llm_generate(prompt: str):
    try:
        logger.info(f"Generating response for prompt: {prompt}")
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        content_response = response.choices[0].message.content
        logger.info(f"Generated response: {content_response}")
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [ ]:
app = FastAPI()

@app.get("/")
def read_root():
    return {"Hello": "World"}

@app.post("/generate")
def generate_response(prompt: str):
    try:
        response = llm_generate(prompt)
        return JSONResponse(content={"response": response})
    except Exception as e:
        print(f"Error generating response: {e}")
        return {"error": "Failed to generate response"}
        

In [ ]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run)
thread.start()

In [ ]:
# To stop the server, you can run the following command in your terminal:
#!kill $(lsof -t -i :8000)